In [ ]:
from locale import DAY_1
import torch
import torch.functional as F
from torch.utils.data import TensorDataset, DataLoader

from triton.profiler import data
from triton.python.triton.tools.triton_to_gluon_translator.scoped_dict import K


def multi_head_attention(x, W_qkv, W_o, n_heads):
    # x:      [B, T, d]        input hidden states
    # W_qkv:  [d, 3d]          fused QKV projection
    # W_o:    [d, d]           output projection
    # n_heads: int             d must be divisible by n_heads
    # returns [B, T, d]
    B, T, d = x.shape
    d_head = d // n_heads

    Q, K, V = torch.split(x @ W_qkv, [d, d, d], dim=-1)

    Q = Q.view(B, T, n_heads, d_head).transpose(1, 2)
    K = K.view(B, T, n_heads, d_head).transpose(1, 2)
    V = V.view(B, T, n_heads, d_head).transpose(1, 2)

    attention_scores = Q @ K.transpose(-1, -2) / d_head**0.5
    mask = torch.tril(torch.ones(T, T, device=x.device, dtype=bool))
    masked_attention_scores = attention_scores.masked_fill(~mask, float("-inf"))

    attention_weights = F.softmax(masked_attention_scores, dim=-1)
    A = attention_weights @ V

    A = A.transpose(1, 2).contiguous().view(B, T, d)

    return A @ W_o


def mlp(x, W_in, W_out):
    # x:     [B, T, d]
    # W_in:  [d, d_ff]
    # W_out: [d_ff, d]
    # returns [B, T, d]      # use GELU
    return F.gelu(x @ W_in) @ W_out


def transformer_block(
    x, W_qkv, W_o, W_in, W_out, gamma1, beta1, gamma2, beta2, n_heads
):
    # gamma/beta are LayerNorm params, shape [d]
    # reuse your multi_head_attention from Q2
    # returns [B, T, d]
    eps = 1e-6
    y = beta1 + gamma1 * (x - x.mean(dim=-1, keepdim=True)) / torch.sqrt(
        x.var(dim=-1, keepdim=True, unbiased=False) + eps
    )
    y = x + multi_head_attention(y, W_qkv, W_o, n_heads)
    z = beta2 + gamma2 * (y - y.mean(dim=-1, keepdim=True)) / torch.sqrt(
        y.var(dim=-1, keepdim=True, unbiased=False) + eps
    )
    z = y + mlp(z, W_in, W_out)
    return z


def gpt(idx, embed, pos_embed, blocks_params, ln_f, n_heads):
    # idx:          [B, T]                token ids
    # embed:        [V, d]               token embedding table
    # pos_embed:    [max_T, d]           learned positional embeddings
    # blocks_params: list of per-block param tuples for transformer_block
    # ln_f:         (gamma, beta) final layernorm
    # returns logits [B, T, V]           (weight-tied output: reuse embed)
    eps = 1e-6
    input = embed[idx]
    B, T = idx.shape
    input = input + pos_embed[:T, :]
    for block_params in blocks_params:
        input = transformer_block(input, *block_params, n_heads)
    input = ln_f.beta + ln_f.gamma * (
        input - input.mean(dim=-1, keepdim=True)
    ) / torch.sqrt(input.var(dim=-1, keepdim=True, unbiased=False) + eps)
    return input @ embed.T


def get_batch(data, batch_size, block_size):
    # data: 1D LongTensor of token ids
    # return: x, y each [batch_size, block_size]
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + 1 + block_size] for i in ix])
    return x, y


def train(model_params, data, n_steps, lr, batch_size, block_size, vocab_size):
    # model_params: everything gpt() needs, as trainable tensors
    # data:         1D LongTensor of token ids (the full training corpus)
    # samples random (x, y) batches, computes next-token CE loss,
    # backprops, and updates params with AdamW.
    # return: list of per-step losses
    optim = torch.optim.AdamW(params=model_params, lr=lr)
    losses = []
    for _ in range(n_steps):
        x, y = get_batch(data, batch_size, block_size)
        optim.zero_grad()
        logits = gpt(x, *model_params)
        loss = F.cross_entropy(logits.view(-1, logits.shape[-1]), y.view(-1))
        losses.append(loss.item())
        loss.backward()
        optim.step()
    return losses


def decode(logits, strategy, temperature=1.0, k=None, p=None):
    # logits: [V]  raw next-token logits for a single position
    # strategy: one of "greedy", "temperature", "top_k", "top_p"
    # returns: an int token id (sampled or argmax)
    if strategy == "greedy":
        return torch.argmax(logits).item()
    elif strategy == "temperature":
        probs = F.softmax(logits / temperature, dim=-1)
        return torch.multinomial(probs, 1).item()
    elif strategy == "topk":
        kth = torch.topk(logits, k).values[-1]
        logits = logits.masked_fill(logits < kth, float("-inf"))
        probs = F.softmax(logits, dim=-1)
        return torch.multinomial(probs, 1).item()
    elif strategy == "topp":
        probs = F.softmax(logits, dim=-1)
        sorted_probs, _ = torch.sort(probs, descending=True, dim=-1)
        cum_probs = torch.cumsum(sorted_probs, dim=-1)
        cutoff = sorted_probs[(cum_probs < p).sum()]
        probs[probs < cutoff] = 0
        return torch.multinomial(probs / probs.sum(), 1).item()


def mlp_forward(x, W1, b1, W2, b2):
    # x:  [B, d_in]
    # W1: [d_in, d_h],  b1: [d_h]
    # W2: [d_h, d_out], b2: [d_out]
    # h = relu(x @ W1 + b1); y = h @ W2 + b2
    # return y and whatever cache you need for backward
    a = x @ W1 + b1
    h = torch.relu(a)
    y = h @ W2 + b2
    return y, {"W2": W2, "W1": W1, "h": h, "a": a, "x": x}


def mlp_backward(dy, cache):
    # dy: [B, d_out]  upstream gradient dL/dy
    # return: dx, dW1, db1, dW2, db2
    db2 = dy.sum(dim=0)
    dh = dy @ cache["W2"].T
    dW2 = cache["h"].T @ dy
    da = dh * (cache["a"] > 0)
    db1 = da.sum(dim=0)
    dW1 = cache["x"].T @ da
    dx = da @ cache["W1"].T
    return dx, dW1, db1, dW2, db2


def mlp2_forward(x, W1, b1, W2, b2, W3, b3):
    # x:  [B, d]
    # W1: [d, d],  b1: [d]     (square, so residuals line up)
    # W2: [d, d],  b2: [d]
    # W3: [d, d],  b3: [d]
    #

    # return y and cache
    a1 = x @ W1 + b1
    h1 = torch.relu(a1)
    z1 = h1 + x
    a2 = z1 @ W2 + b2
    h2 = torch.relu(a2)
    z2 = h2 + z1
    y = z2 @ W3 + b3
    return y, {
        "x": x,
        "W1": W1,
        "a1": a1,
        "h1": h1,
        "z1": z1,
        "a2": a2,
        "h2": h2,
        "z2": z2,
        "y": y,
    }


def mlp2_backward(dy, cache):
    # dy: [B, d]  upstream grad dL/dy
    # return: dx, dW1, db1, dW2, db2, dW3, db3
    dz2 = dy @ cache["W3"].T
    dW3 = cache["z2"].T @ dy
    db3 = dy.sum(dim=0)

    dh2 = dz2
    dz1 = dz2

    da2 = dh2 * (cache["a2"] > 0)
    dz1 += da2 @ cache["W2"].T
    dW2 = cache["z1"].T @ da2
    db2 = da2.sum(dim=0)

    dh1 = dz1
    dx = dz1

    da1 = dh1 * (cache["a1"] > 0)
    dW1 = cache["x"].T @ da1
    db1 = da1.sum(dim=0)
    dx += da1 @ cache["W1"].T

    return dx, dW1, db1, dW2, db2, dW3, db3


def attention_forward(Q, K, V):
    # Q, K, V: [T, d]
    # return out, cache
    T, d = Q.shape
    attention_scores = Q @ K.T / d**0.5
    mask = torch.tril(torch.ones(T, T, dtype=bool))
    masked_attention_scores = attention_scores.masked_fill(~mask, float("-inf"))
    attention_weights = F.softmax(masked_attention_scores, dim=-1)
    out = attention_weights @ V
    return out, {
        "out": out,
        "attention_weights": attention_weights,
        "masked_attention_scores": masked_attention_scores,
        "attention_scores": attention_scores,
        "Q": Q,
        "K": K,
        "V": V,
    }


def attention_backward(d_out, cache):
    # d_out: [T, d]
    # return dQ, dK, dV
    T, d = d_out.shape
    d_attention_weights = d_out @ cache["V"].T
    d_V = cache["attention_weights"].T @ d_out
    d_masked_attention_scores = cache["attention_weights"] * (
        d_attention_weights
        - (d_attention_weights * cache["attention_weights"]).sum(dim=-1, keepdim=True)
    )
    d_attention_scores = d_masked_attention_scores
    d_Q = d_attention_scores @ cache["K"] / d**0.5
    d_K = (cache["Q"].T @ d_attention_scores).T / d**0.5
    return d_Q, d_K, d_V

In [ ]:
def flash_attention_forward(Q, K, V, block_size):
    # Q, K, V: [B, S, d]
    # block_size: keys/values per tile
    # returns out: [B, S, d]
    B, S, d = Q.shape
    scale = d ** -0.5

    Q_blocks = torch.split(Q, block_size, dim=-2)
    K_blocks = torch.split(K, block_size, dim=-2)
    V_blocks = torch.split(V, block_size, dim=-2)
    O_blocks = []

    for q in Q_blocks:
        m = torch.full((B, q.shape[-2], 1), float("-inf"))
        l = torch.zeros(B, q.shape[-2], 1)
        o = torch.zeros(B, q.shape[-2], d)

        for k, v in zip(K_blocks, V_blocks):
            s = q @ k.transpose(-1, -2) * scale
            new_m = torch.max(m, torch.max(s, dim=-1, keepdim=True).values)
            m_corr = torch.exp(m - new_m)
            P = torch.exp(s - new_m)
            new_l = l * m_corr + P.sum(dim=-1, keepdim=True)
            new_o = o * m_corr + P @ v

            m = new_m
            l = new_l
            o = new_o

        O_blocks.append(o / l)

    return torch.cat(O_blocks, dim=-2)


